# ARGUS Phase 3 — Classifier Evaluation & Threshold Calibration

Evaluates the fine-tuned classifier on **same-day normal controls** (not just val split).
Produces ROC-AUC, PR-AUC, threshold sweep, confusion matrices, and real alert-engine outputs.

**Prerequisites:** Run the Phase 2.5 + Phase 3 fine-tune notebooks first.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, json, time

# ── Paths ──
DATA_ROOT   = Path("/kaggle/input/datasets/nightingale21/argus-tokenized-58day-verified/data")
VOCAB_PATH  = DATA_ROOT / "vocab.json"
if not VOCAB_PATH.exists():
    VOCAB_PATH = DATA_ROOT / "tokenized" / "vocab.json"
SESSIONS_DIR = DATA_ROOT / "sessions"           # day_XX.parquet files
VAL_MANIFEST = DATA_ROOT / "tokenized" / "sessions_val.pt"

REDTEAM_PATH = Path("/kaggle/input/datasets/nightingale21/attacker/redteam.txt")

REPO_URL = "https://github.com/NIghtIngale340/ARGUS"
REPO_DIR = Path("/kaggle/working/argus-log-intelligence-platform")
REFRESH_REPO = True

# Phase 2.5 outputs
ATTACK_MANIFEST = Path("/kaggle/working/attack_sessions/attack_sessions.pt")

# Phase 3 fine-tune outputs
CLASSIFIER_CKPT = Path("/kaggle/working/argus_finetuned/best_classifier.pt")

# Phase 3 eval outputs
EVAL_OUT = Path("/kaggle/working/argus_phase3_eval")
EVAL_OUT.mkdir(parents=True, exist_ok=True)

# Attack days (from Phase 2.5 scan)
ATTACK_DAYS = [
    2, 3, 6, 7, 8, 9, 10, 13, 14, 15, 16,
    21, 22, 23, 27, 28, 29, 30
]


In [ ]:
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

def run_stream(command, cwd=None, env=None):
    print("$", " ".join(str(p) for p in command), flush=True)
    proc = subprocess.Popen(
        [str(p) for p in command], cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc})")

if REFRESH_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    run_stream(["git", "clone", REPO_URL, str(REPO_DIR)])

run_stream([sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.37.0", "pyarrow>=14.0.0", "tqdm>=4.67.1"])

sys.path.insert(0, str(REPO_DIR))
print("Repo ready:", REPO_DIR)


## Step 1: Verify prerequisites

In [ ]:
required = {
    "Classifier": CLASSIFIER_CKPT,
    "Attack manifest": ATTACK_MANIFEST,
    "Val manifest": VAL_MANIFEST,
    "Vocab": VOCAB_PATH,
    "Redteam": REDTEAM_PATH,
}
for name, p in required.items():
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status}: {name} → {p}")
    if not p.exists():
        raise FileNotFoundError(f"{name} not found: {p}")

# Load classifier info
ckpt = torch.load(CLASSIFIER_CKPT, map_location="cpu", weights_only=False)
print(f"\nClassifier: epoch {ckpt.get('epoch')}, train F1={ckpt.get('best_f1', 0):.4f}")
print(f"Val metrics: {ckpt.get('val_metrics', {})}")


## Step 2: Build same-day normal controls

Attack sessions come from days 02–30. For a fair evaluation, our normal controls
should also come from those same days (not from the temporal val split days 41+).

We load session parquets for attack days, remove attack sessions, tokenize a sample
of normal sessions, and save as a manifest.

In [ ]:
import bisect
import csv
from collections.abc import Mapping

import pyarrow.parquet as pq
import torch
from src.parsing.log_tokenizer import LogTokenizer

CONTROL_DIR = EVAL_OUT / "normal_controls"
CONTROL_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_MANIFEST = CONTROL_DIR / "normal_controls.pt"

MAX_SEQ_LEN = 16
MAX_NORMAL_PER_DAY = 1000
PARQUET_BATCH_SIZE = 20000
ATTACK_LABELS_CSV = Path("/kaggle/working/attack_sessions/attack_session_labels.csv")

tokenizer = LogTokenizer(str(VOCAB_PATH), max_len=MAX_SEQ_LEN)
print(f"Vocab: {len(tokenizer.vocab)} tokens, max_len={tokenizer.max_len}")

attack_windows = {}
if ATTACK_LABELS_CSV.exists():
    with ATTACK_LABELS_CSV.open(newline="") as f:
        for row in csv.DictReader(f):
            key = (row.get("user_id", ""), row.get("host_id", ""))
            attack_windows.setdefault(key, []).append((int(row.get("start_ts", 0)), int(row.get("end_ts", 0))))
    for key in attack_windows:
        attack_windows[key].sort()
print(f"Attack windows loaded: {sum(len(v) for v in attack_windows.values()):,}")

def coerce_events(events):
    if not isinstance(events, list):
        if hasattr(events, "tolist"):
            events = events.tolist()
        elif isinstance(events, tuple):
            events = list(events)
        else:
            return []
    return [dict(event) for event in events if isinstance(event, Mapping)]

def overlaps_attack(row):
    if not attack_windows:
        return False
    key = (row.get("user_id", ""), row.get("host_id", ""))
    windows = attack_windows.get(key)
    if not windows:
        return False
    st, et = int(row.get("start_ts", 0)), int(row.get("end_ts", 0))
    starts = [window[0] for window in windows]
    pos = bisect.bisect_right(starts, et)
    return any(win_end >= st for _, win_end in windows[:pos])

normal_sessions = []
total_scanned = 0
total_skipped_attack = 0

for day_num in ATTACK_DAYS:
    parquet_path = SESSIONS_DIR / f"day_{day_num:02d}.parquet"
    if not parquet_path.exists():
        print(f"  day_{day_num:02d}: not found, skipping")
        continue

    pf = pq.ParquetFile(parquet_path)
    available_columns = set(pf.schema_arrow.names)
    wanted_columns = [
        c for c in ["session_id", "user_id", "host_id", "start_ts", "end_ts", "events"]
        if c in available_columns
    ]
    day_count = 0

    for batch in pf.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=wanted_columns):
        for row in batch.to_pylist():
            total_scanned += 1
            if overlaps_attack(row):
                total_skipped_attack += 1
                continue
            events = coerce_events(row.get("events", []))
            if not events:
                continue
            session = dict(row)
            session["events"] = events
            normal_sessions.append(session)
            day_count += 1
            if day_count >= MAX_NORMAL_PER_DAY:
                break
        if day_count >= MAX_NORMAL_PER_DAY:
            break

    print(f"  day_{day_num:02d}: {day_count} normal control session(s)")

print(f"\nTotal scanned: {total_scanned:,}")
print(f"Attack-overlap sessions skipped: {total_skipped_attack:,}")
print(f"Normal controls selected: {len(normal_sessions):,}")


In [ ]:
if normal_sessions:
    stats = tokenizer.save_tokenized_sessions_pt_chunked_with_stats(
        sessions=normal_sessions,
        output_path=CONTROL_MANIFEST,
        chunk_size=5000,
        token_id_dtype=torch.int16,
        attention_mask_dtype=torch.bool,
    )
    print(f"Normal controls manifest: {CONTROL_MANIFEST}")
    print(f"  {stats.session_count:,} sessions, {stats.chunk_count} chunk(s)")
else:
    raise RuntimeError("No normal controls built; check session parquet paths and event columns.")


## Step 3: Run classifier evaluation

In [ ]:
# Determine which normal manifest to use
normal_manifest = str(CONTROL_MANIFEST) if CONTROL_MANIFEST.exists() else str(VAL_MANIFEST)

eval_cmd = [
    sys.executable, "-m", "scripts.evaluate_attack_classifier",
    "--classifier", str(CLASSIFIER_CKPT),
    "--attack-manifest", str(ATTACK_MANIFEST),
    "--normal-manifest", normal_manifest,
    "--out", str(EVAL_OUT),
    "--max-normal", "10000",
    "--batch-size", "128",
    "--num-workers", "2",
]
run_stream(eval_cmd, cwd=REPO_DIR,
           env={**os.environ, "PYTHONPATH": str(REPO_DIR)})


## Step 4: Review results

In [ ]:
report_path = EVAL_OUT / "evaluation_report.json"
report = json.loads(report_path.read_text())

print("=" * 60)
print("ARGUS Phase 3 - Classifier Evaluation Report")
print("=" * 60)
print(f"\nAttack sessions:  {report['n_attack']}")
print(f"Normal controls:  {report['n_normal']}")
print(f"\nROC-AUC:  {report['roc_auc']:.6f}")
print(f"PR-AUC:   {report['pr_auc']:.6f}")

print(f"\nAttack P(attack): mean={report['attack_prob_stats']['mean']:.4f} "
      f"std={report['attack_prob_stats']['std']:.4f}")
print(f"Normal P(attack): mean={report['normal_prob_stats']['mean']:.4f} "
      f"std={report['normal_prob_stats']['std']:.4f}")

best_f1 = report["best_f1_threshold"]
operating = report["operating_threshold"]
print(f"\nBest F1 threshold: {best_f1['threshold']:.4f}")
print(f"  F1={best_f1['f1']:.4f} Prec={best_f1['precision']:.4f} "
      f"Rec={best_f1['recall']:.4f} FPR={best_f1['fpr']:.4f}")
print(f"\nOperating threshold: {operating['threshold']:.4f}")
print(f"  F1={operating['f1']:.4f} Prec={operating['precision']:.4f} "
      f"Rec={operating['recall']:.4f} FPR={operating['fpr']:.4f}")

print("\n-- Threshold Sweep --")
print(f"{'Thresh':>8} {'Prec':>8} {'Recall':>8} {'F1':>8} {'FPR':>8}")
for row in report["threshold_sweep"]:
    print(f"{row['threshold']:>8.4f} {row['precision']:>8.4f} {row['recall']:>8.4f} "
          f"{row['f1']:>8.4f} {row['fpr']:>8.4f}")

print("\n-- Alert Engine (operating threshold) --")
ae = report["alert_engine_results"]
print(f"  Threshold: {ae['classification_threshold']:.4f}")
print(f"  Alerts:    {ae['total_alerts']}")
print(f"  True pos:  {ae['true_attack_alerts']}")
print(f"  False pos: {ae['false_alerts']}")
print(f"  Precision: {ae['alert_precision']:.4f}")
print(f"  Severity:  {ae['severity_distribution']}")


## Step 5: Archive

In [ ]:
ARCHIVE = Path("/kaggle/working/argus_phase3_eval_archive")
ARCHIVE.mkdir(exist_ok=True)

for f in [
    EVAL_OUT / "evaluation_report.json",
    EVAL_OUT / "classifier_scores.csv",
    EVAL_OUT / "calibrated_thresholds.json",
    CLASSIFIER_CKPT,
    Path("/kaggle/working/argus_finetuned/finetune_history.json"),
]:
    if f.exists():
        shutil.copy2(f, ARCHIVE / f.name)
        print(f"  Archived: {f.name}")

archive = shutil.make_archive(str(ARCHIVE), "zip", root_dir=ARCHIVE)
print(f"\nArchive: {archive}")
print("Phase 3 evaluation complete!")


## Step 6: Package Phase 3 model bundle

Package the trained classifier, calibrated thresholds, vocab, eval outputs, and metadata into a reusable bundle.


In [ ]:
BASE_MLM_CKPT = Path("/kaggle/working/argus_mlm_eval_check/checkpoint_step_003501.pt")
BUNDLE_DIR = Path("/kaggle/working/argus_phase3_model_bundle")
FINETUNE_HISTORY = Path("/kaggle/working/argus_finetuned/finetune_history.json")

# Make sure Kaggle is using the pushed repo that contains the packaging/detection scripts.
run_stream(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR)

bundle_required = {
    "Classifier checkpoint": CLASSIFIER_CKPT,
    "Calibrated thresholds": EVAL_OUT / "calibrated_thresholds.json",
    "Vocabulary": VOCAB_PATH,
    "Evaluation report": EVAL_OUT / "evaluation_report.json",
    "Classifier scores": EVAL_OUT / "classifier_scores.csv",
    "Fine-tune history": FINETUNE_HISTORY,
    "Base MLM checkpoint": BASE_MLM_CKPT,
}
for name, path in bundle_required.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"  {status}: {name} -> {path}")
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")

package_cmd = [
    sys.executable, "-m", "scripts.package_phase3_model_bundle",
    "--classifier", str(CLASSIFIER_CKPT),
    "--thresholds", str(EVAL_OUT / "calibrated_thresholds.json"),
    "--vocab", str(VOCAB_PATH),
    "--evaluation-report", str(EVAL_OUT / "evaluation_report.json"),
    "--classifier-scores", str(EVAL_OUT / "classifier_scores.csv"),
    "--finetune-history", str(FINETUNE_HISTORY),
    "--base-checkpoint", str(BASE_MLM_CKPT),
    "--out-dir", str(BUNDLE_DIR),
]
run_stream(package_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})

metadata = json.loads((BUNDLE_DIR / "model_metadata.json").read_text())
print("\nBundle metadata check")
print("  format:", metadata["bundle_format"])
print("  threshold:", metadata["operating_threshold"])
print("  vocab_size:", metadata["vocab_size"])
print("  max_seq_len:", metadata["max_seq_len"])
print("  archive:", Path(str(BUNDLE_DIR) + ".zip"))


## Step 7: Run tiny bundle inference smoke

Run one JSONL session through the packaged bundle to verify the reusable detection path.


In [ ]:
SMOKE_JSONL = Path("/kaggle/working/argus_phase3_smoke_sessions.jsonl")
SMOKE_OUT = Path("/kaggle/working/argus_phase3_smoke_detection.csv")

smoke_sessions = [
    {
        "session_id": "phase3_smoke_001",
        "user_id": "U_smoke",
        "host_id": "C_smoke",
        "events": [
            {"event_id": "NA", "auth_type": "NTLM", "logon_type": "Network"},
            {"event_id": "NA", "auth_type": "Kerberos", "logon_type": "Interactive"},
            {"event_id": "NA", "auth_type": "NTLM", "logon_type": "Network"},
        ],
    }
]

with SMOKE_JSONL.open("w", encoding="utf-8") as f:
    for row in smoke_sessions:
        f.write(json.dumps(row) + "\n")

detect_cmd = [
    sys.executable, "-m", "scripts.run_detection",
    "--bundle-dir", str(BUNDLE_DIR),
    "--sessions-jsonl", str(SMOKE_JSONL),
    "--out", str(SMOKE_OUT),
    "--batch-size", "16",
]
run_stream(detect_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": str(REPO_DIR)})

print("\nSmoke output:")
print(SMOKE_OUT.read_text())
